# CNN
[参考链接](https://zhuanlan.zhihu.com/p/630695553)

In [1]:
import torch
import torch.nn as nn
import torch.utils.data as Data
from torch.autograd import Variable
import torchvision # pytorch的一个视觉处理工具包(需单独安装)

# 关于torchvision中的数据集

* torchvision中datasets中所有封装的数据集都是torch.utils.data.Dataset的子类，它们都可以用torch.utils.data.DataLoader进行数据加载。
> 以datasets.MNIST类为例，具体参数和用法如下所示：

```python
CLASS torchvision.datasets.MNIST(
          root: str, 
          train: bool = True, 
          transform: Optional[Callable] = None, 
          target_transform: Optional[Callable] = None, 
          download: bool = False
)
```
## 参数解释
* **root (string)：** 表示数据集的根目录，其中根目录存在MNIST/processed/training.pt和MNIST/processed/test.pt的子目录**(其实就是对下载的文件指定位置)**
* **train (bool, optional)：** 如果为True，则从training.pt创建数据集，否则从test.pt创建数据集。**（想要的是训练集还是测试集）**
* **download (bool, optional)：** 如果为True，则从internet下载数据集并将其放入根目录。如果数据集已下载，则不会再次下载
* **transform (callable, optional)：** 接收PIL图片并返回转换后版本图片的转换函数 **(就是把图片或者numpy中的数组转换成tensor)**
* **target_transform (callable, optional)：** 接收PIL接收目标并对其进行变换的转换函数


# Variable and Tensor
[参考链接](https://blog.csdn.net/weixin_44912159/article/details/104800020)

# Step 2 数据预处理

In [3]:
torch.manual_seed(1) # 设置随机种子，用于复现

# 超参数
EPOCH = 1 # 前向后向传播迭代次数
LR = 0.001 # 学习率 learning rate
BATCH_SIZE = 50 # 批量训练时候一次送入的数据的size
DOWNLOAD_MNIST = True

# 下载mnist手写数据集
# 训练集
train_data = torchvision.datasets.MNIST(
    root = './MNIST/',
    train = True,
    transform = torchvision.transforms.ToTensor(),
    # download=DOWNLOAD_MNIST
    download=False
)

# 测试集
test_data = torchvision.datasets.MNIST(root='./MNIST/',train=False) 
# train设置为False表示获取测试集

# 一个批训练50个样本，1 channel通道（灰色）,图片尺寸 28x28 size:(50, 1, 28, 28)
train_loader = Data.DataLoader(
    dataset=train_data,
    batch_size=BATCH_SIZE,
    shuffle=True
)
# 测试数据预处理：只测试前2000个
# 通过在第1个维度上添加一个维度，将每个图像从 (28, 28) 转换为 (1, 28, 28)，使其变成一个单通道的图像
test_x = torch.unsqueeze(test_data.data, dim = 1).float()[:2000] / 255.0
# shape from (2000, 28, 28) to (2000, 1, 28, 28)
test_y = test_data.targets[:2000]

# Illustration for Step 2
* Data.DataLoader:加载数据
    * shuffle:表示打乱数据顺序
* torch.unsqueeze：作者理解就是改变数据shape，此处就是把训练数据本来是一维的给"竖"起来作为一条一条数据进行训练(具体函数用法看下面参考资料)
    * [参考1](https://link.zhihu.com/?target=https%3A//blog.csdn.net/flyingluohaipeng/article/details/125092937)[参考2](https://zhuanlan.zhihu.com/p/86763381)
    * 图像尺寸是28*28的，具体验证可见最下面作者后续转成的html结果

# Step 3 定义网络结构

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        
        self.conv1 = nn.Sequential(
            nn.Conv2d(  # 输入的图片 (1,28,28)
                in_channels=1,
                out_channels=16, # 经过一个卷积层之后(16,28,28)
                kernel_size=5,
                stride=1, # res_w = (m_w - kernel_size + 2*padding)/stride + 1
                padding=2
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2) # 经过池化层处理，维度为（16，14，14）
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d( # 输入为（16，14，14）
                in_channels=16,
                out_channels=32,
                kernel_size=5,
                stride=1,
                padding=2
            ), # 输出为（32，14，14）
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2) # 输出为（32，7，7）
        )
        
        self.out = nn.Linear(in_features=32*7*7,out_features=10)
        
    def forward(self, x): # x :(batch_size, 1, 28, 28)
        x = self.conv1(x)  # 输出 (batch_size, 16, 14, 14)
        x = self.conv2(x)  # 输出 (batch_size, 32, 7, 7)
        x = x.view(x.size(0),-1)  # (batch_size, 32*7*7)输出为batch_size行的矩阵，每一行对应32*7*7个参数
        out = self.out(x)  # (batch_size, 10) init的时候设置好了input参数了,out_features也代表了该全连接层的神经元个数。
        return out

cnn = CNN()

## 一些不清楚的地方的对应解释
* nn.Linear主要是用于全连接层[参考链接](https://link.zhihu.com/?target=https%3A//blog.csdn.net/qq_42079689/article/details/102873766)
    * nn.Linear具体用法(10是因为这个识别结果是0-9，为10个类别)：[参考链接](https://link.zhihu.com/?target=https%3A//www.cnblogs.com/douzujun/p/13366939.html)
    > 用于设置网络中的全连接层，需要注意的是全连接层的输入与输出都是二维张量
* x.view()就是对tensor进行reshape：[参考链接](https://link.zhihu.com/?target=https%3A//blog.csdn.net/echo_gou/article/details/121035061)

# Step 4 训练模型

In [5]:
optimizer = torch.optim.Adam(cnn.parameters(),lr=LR)  # 定义优化器
loss_func = nn.CrossEntropyLoss()  # 定义损失函数

for epoch in range(EPOCH):
    for step,(batch_x,batch_y) in enumerate(train_loader):
        pred_y = cnn(batch_x)
        loss = loss_func(pred_y, batch_y)
        optimizer.zero_grad()  # 清空上一层梯度
        loss.backward()  # 反向传播
        optimizer.step()  # 更新优化器的学习率，一般按照epoch为单位进行更新
        
        if step%50 == 0:
            test_output = cnn(test_x)
            pred_y = torch.max(test_output,1)[1].numpy()
            print('Epoch: ', epoch, '| train loss: %.4f' % loss.data.numpy())
            

test_output = cnn(test_x[:10])
pred_y = torch.max(test_output, 1)[1].numpy()  # 经过全连接层之后输出通常是一个包含每个类别的分数或概率的向量，因此选取最大值为概率最大的label
print(pred_y, 'prediction number')
print(test_y[:10].numpy(), 'real number')

Epoch:  0 | train loss: 2.3125
Epoch:  0 | train loss: 0.4850
Epoch:  0 | train loss: 0.3738
Epoch:  0 | train loss: 0.3814
Epoch:  0 | train loss: 0.2352
Epoch:  0 | train loss: 0.1629
Epoch:  0 | train loss: 0.0554
Epoch:  0 | train loss: 0.1943
Epoch:  0 | train loss: 0.0662
Epoch:  0 | train loss: 0.0980
Epoch:  0 | train loss: 0.0842
Epoch:  0 | train loss: 0.0603
Epoch:  0 | train loss: 0.0407
Epoch:  0 | train loss: 0.0887
Epoch:  0 | train loss: 0.0197
Epoch:  0 | train loss: 0.0876
Epoch:  0 | train loss: 0.2289
Epoch:  0 | train loss: 0.0396
Epoch:  0 | train loss: 0.0432
Epoch:  0 | train loss: 0.0823
Epoch:  0 | train loss: 0.0117
Epoch:  0 | train loss: 0.0870
Epoch:  0 | train loss: 0.0267
Epoch:  0 | train loss: 0.0577
[7 2 1 0 4 1 4 9 5 9] prediction number
[7 2 1 0 4 1 4 9 5 9] real number


# Additionally 
如果已经有数据集，读取的话参考下面的链接
[参考链接](https://zhuanlan.zhihu.com/p/306399851)